In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["Groq_Api_key"] = os.getenv("Groq_Api_key")

model = init_chat_model("groq:qwen/qwen3-32b")


f:\Python\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from pydantic import BaseModel, Field


class Movie(BaseModel):
    title:str=Field(description="Title of the movie")
    year:int=Field(description="This year movie was released")
    director:str=Field(description="Person who directed the movie")
    rating:float=Field(description="Movie's  IMDB rating out of 10")
    

In [5]:
model_with_structure = model.with_structured_output(Movie)

model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002C506903690>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002C5069B5050>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'Title of the movie', 'type': 'string'}, 'year': {'description': 'This year movie was released', 'type': 'integer'}, 'director': {'description': 'Person who directed the movie', 'type': 'string'}, 'rating': {'description': "Movie's  IMDB rating out of 10", 'type': 'number'}}, 'required': ['title', 'year', 'd

In [6]:
response = model_with_structure.invoke("Tell be about the Godfather")

In [7]:
response

Movie(title='The Godfather', year=1972, director='Francis Ford Coppola', rating=9.2)

In [8]:
# Messaged Output along with parsed structre

model_with_structure = model.with_structured_output(Movie, include_raw = True)
response = model_with_structure.invoke("Tell be about the Godfather")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user asked me to tell them about The Godfather. Let me think about how to approach this. They probably want a summary of the movie, maybe some key details like the director, release year, and rating.\n\nFirst, I need to check if there\'s a function available to get this information. The provided tools include a Movie function with parameters for director, rating, title, and year. So I should use that.\n\nI should call the Movie function with the title "The Godfather". I need to make sure the arguments are correctly formatted. The required fields are title, year, director, and rating. Let me recall the details: The Godfather was directed by Francis Ford Coppola, released in 1972, and has a high IMDB rating, maybe 9.2.\n\nWait, I should confirm the exact details. But since I don\'t have a database, I\'ll rely on my existing knowledge. The function will return the structured data, which I can then use to form

In [11]:
class Actor(BaseModel):
    name:str
    role:str
    
class MovieDetails(BaseModel):
    title:str
    year:int
    cast: list[Actor]
    generes:list[str]
    budget:float| None = Field(None, description="Budget in million USD")
    
model_with_structured_output = model.with_structured_output(MovieDetails)
response = model_with_structured_output.invoke("Provide detals of the movie Atonement")
response

    

MovieDetails(title='Atonement', year=2007, cast=[Actor(name='Keira Knightley', role='Briony'), Actor(name='James McAvoy', role='Robbie'), Actor(name='Vanessa Redgrave', role='Lola')], generes=['Drama', 'Romance', 'War'], budget=15.0)

In [13]:
from typing_extensions import TypedDict, Annotated

class Actor(TypedDict):
    name:str
    role:str
    
class MovieDetails(TypedDict):
    title:str
    year:int
    cast: list[Actor]
    generes:list[str]
    budget:float| None = Field(None, description="Budget in million USD")
    
model_with_typedict_output = model.with_structured_output(MovieDetails)
response = model_with_typedict_output.invoke("Provide detals of the movie Atonement")
response


{'budget': 15000000,
 'cast': [{'name': 'Keira Knightley', 'role': 'Briony'},
  {'name': 'James McAvoy', 'role': 'Robbie'},
  {'name': 'Saoirse Ronan', 'role': 'Cecilia'}],
 'generes': ['Drama', 'Romance', 'War'],
 'title': 'Atonement',
 'year': 2007}

DataClasses

In [ ]:
DataClasses